In [ ]:
!pip install -q -U transformers datasets peft accelerate pandas scikit-learn torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 150.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 155.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.


In [ ]:
from google.colab import userdata, drive
from huggingface_hub import login
import torch
import pandas as pd
import os
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import get_peft_model, LoraConfig, TaskType

# ==========================================
# 1. SETUP & AUTHENTICATION
# ==========================================
login(token=userdata.get('HF_TOKEN'))

PROJECT_DIR = '/content'
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
DATA_FILE = f"{PROJECT_DIR}/medhallu_lora_training_data_batch.csv"
OUTPUT_DIR = f"{PROJECT_DIR}/llama3-medhallu-router"

# ==========================================
# 2. PREPARE THE DATASET
# ==========================================
print("Loading Training Data...")

# Bulletproof CSV Loading
try:
    df = pd.read_csv(DATA_FILE, lineterminator='\n')
except Exception:
    print("Strict parsing failed. Falling back to forgiving Python parser...")
    df = pd.read_csv(DATA_FILE, engine='python', on_bad_lines='skip')

print(f"Successfully loaded {len(df)} rows!")

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # UPGRADED FOR A100: Max context length 2048
    return tokenizer(examples["prompt"], truncation=True, max_length=2048)

print("Tokenizing Dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ==========================================
# 3. INJECT LORA ADAPTER
# ==========================================
print("Loading Base Model for Sequence Classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16
)
model.config.pad_token_id = tokenizer.eos_token_id

print("Injecting LoRA parameters...")
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ==========================================
# 4. TRAIN THE ROUTER
# ==========================================
# IMPORTANT: Disable caching when using gradient checkpointing
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,

    # --- THE VRAM SAVERS ---
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4, # Effective batch size is now 16 (4x4)
    gradient_checkpointing=True,   # Throws away intermediate memory
    per_device_eval_batch_size=4,
    # -----------------------

    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=25, # Lowered slightly so you see updates faster
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("\nStarting Phase 2 LoRA Training...")
trainer.train()

print(f"\nTraining Complete! Saving final adapter weights to {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Phase 2 Complete!")

Loading Training Data...
Successfully loaded 9000 rows!
Loading Tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


trainable params: 6,823,936 || all params: 7,511,756,800 || trainable%: 0.0908

Starting Phase 2 LoRA Training...


`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,2.301992,0.545560
2,2.100430,0.545108



Training Complete! Saving final adapter weights to /content/llama3-medhallu-router
✅ Phase 2 Complete!


In [ ]:
import shutil

# 1. Update this to wherever your model is currently saved locally
local_model_path = "/content/llama3-medhallu-router"

# 2. Your permanent Google Drive destination
drive_destination = "/content/drive/MyDrive/Adaptive_RAG_Project/llama3-medhallu-router"

print("Copying adapter weights to Google Drive. This might take a minute...")

# Copy the entire directory
shutil.copytree(local_model_path, drive_destination, dirs_exist_ok=True)

print(f"✅ Model safely backed up to: {drive_destination}")

Copying adapter weights to Google Drive. This might take a minute...
✅ Model safely backed up to: /content/drive/MyDrive/Adaptive_RAG_Project/llama3-medhallu-router
